### Undisclosed Rebranding
This notebook focuses on undisclosed rebranding of certificates where companies use in their cerificates OpenSSL functions, but do not mention OpenSSL

In [1]:
from sec_certs.dataset.cc import CCDataset
from sec_certs.dataset.fips import FIPSDataset



/home/odin/.local/lib/python3.12/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
print("Downloading CC dataset")

cc_dataset = CCDataset.from_web(auxiliary_datasets=True, artifacts=True, path="./datasets")

print(f"Downloaded {len(cc_dataset)} CC certificates")
print("Saving dataset")

cc_dataset.to_json("./datasets/CC_dataset.json")

print("Dataset saved to ./datasets/CC_dataset.json")

Downloading: CCDataset:   0%|          | 0.00/12.9G [00:00<?, ?B/s]

Dataset was created with sec-certs version 0.4.1.post1.dev86+g770a181a4.d20260805 (newer than your version 0.4.1.post1.dev75+g9e07e5b06). To install the matching version: pip install sec-certs==0.4.1.post1.dev86+g770a181a4.d20260805
Dataset was created with sec-certs version 0.4.1.post1.dev119+g7527f7eac.d20260910 (newer than your version 0.4.1.post1.dev75+g9e07e5b06). To install the matching version: pip install sec-certs==0.4.1.post1.dev119+g7527f7eac.d20260910


Downloaded 6801 CC certificates
Saving dataset
Dataset saved to ./datasets/CC_dataset.json


In [ ]:
print("Downloading FIPS dataset")

fips_dataset = FIPSDataset.from_web()

print(f"Downloaded {len(fips_dataset)} FIPS 140 certificates")
print("Saving dataset")

fips_dataset.to_json("./datasets/FIPS_dataset.json")

print("Dataset saved to ./datasets/FIPS_dataset.json")

In [ ]:
print("Loading datasets from JSON")
cc_dataset = CCDataset.from_json("./datasets/CC_dataset.json")
print(f"Loaded {len(cc_dataset)} CC certificates from JSON")
fips_dataset = FIPSDataset.from_json("./datasets/FIPS_dataset.json")
print(f"Loaded {len(fips_dataset)} FIPS 140 certificates from JSON")

### 1. Load and Filter OpenSSL Functions
We load the list of OpenSSL functions extracted from the docs and filter for specific API prefixes to avoid false positives (e.g., generic words like ).

In [3]:
import json
import re

with open('src/openssl_man3.json', 'r') as f:
    all_functions = json.load(f)

prefixes = ('SSL_', 'EVP_', 'BIO_', 'BN_', 'X509_', 'PEM_', 'CMS_', 'CRYPTO_', 'ASN1_', 'd2i_', 'i2d_')
filtered_functions = [f for f in all_functions if f.startswith(prefixes)]
print(f"Total functions extracted: {len(all_functions)}")
print(f"Filtered high-confidence functions: {len(filtered_functions)}")


Total functions extracted: 5671
Filtered high-confidence functions: 3450


### 2. Define Extraction Logic
We define a function that checks if a certificate mentions OpenSSL via sec-certs keywords. If it doesn't, we scan its associated text files for our filtered OpenSSL functions.

In [4]:
import os
import pandas as pd
from pathlib import Path

def mentions_openssl_cc(cert):
    # Check if OpenSSL is in the CC report or ST keywords
    if cert.pdf_data.report_keywords and 'crypto_library' in cert.pdf_data.report_keywords:
        if 'OpenSSL' in cert.pdf_data.report_keywords['crypto_library']:
            return True
    if cert.pdf_data.st_keywords and 'crypto_library' in cert.pdf_data.st_keywords:
        if 'OpenSSL' in cert.pdf_data.st_keywords['crypto_library']:
            return True
    return False

def mentions_openssl_fips(cert):
    # Check if OpenSSL is in the FIPS policy keywords
    if cert.pdf_data.keywords and 'crypto_library' in cert.pdf_data.keywords:
        if 'OpenSSL' in cert.pdf_data.keywords['crypto_library']:
            return True
    return False

def scan_text_for_functions(text_path, functions):
    if not text_path or not os.path.exists(text_path):
        return {}
    
    with open(text_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
        
    matches = {}
    for func in functions:
        # Use simple string matching for performance
        count = content.count(func)
        if count > 0:
            matches[func] = count
            
    return matches


### 3. Test Extraction Logic on a Subset (Quick Debugging)
To test and debug quickly without scanning all ~12,000 certificates, configure  to scan a smaller slice of the datasets.

In [6]:
SAMPLE_SIZE = 100  # Set to an integer (e.g., 50 or 100) for fast testing

cc_sample = list(cc_dataset)[:SAMPLE_SIZE]
# fips_sample = list(fips_dataset)[:SAMPLE_SIZE]

print(f"Testing logic...")

sample_results = []
for cert in cc_sample:
    if not mentions_openssl_cc(cert):
        r_matches = scan_text_for_functions(cert.state.report_txt_path, filtered_functions)
        st_matches = scan_text_for_functions(cert.state.st_txt_path, filtered_functions)
        matches = {**r_matches}
        for k, v in st_matches.items():
            matches[k] = matches.get(k, 0) + v
        if matches:
            sample_results.append({"Dataset": "CC", "ID": cert.dgst, "Name": cert.name, "MatchedFunctions": list(matches.keys()), "FunctionCount": len(matches)})

#for cert in fips_sample:
 #   if not mentions_openssl_fips(cert):
  #      p_matches = scan_text_for_functions(cert.state.sp.txt_path, filtered_functions)
   #     if p_matches:
    #        sample_results.append({"Dataset": "FIPS", "ID": cert.dgst, "Name": cert.web_data.module_name if hasattr(cert, "web_data") and cert.web_data else None, "MatchedFunctions": list(p_matches.keys()), "FunctionCount": len(p_matches)})

print(f"Subset scan found {len(sample_results)} matches:")
import pandas as pd
display(pd.DataFrame(sample_results))


Testing logic...


AttributeError: 'InternalState' object has no attribute 'report_txt_path'

### 4. Scan Full Common Criteria Dataset

In [ ]:
results = []

print("Scanning CC Dataset...")
for cert in cc_dataset:
    if not mentions_openssl_cc(cert):
        # Check report text
        report_matches = scan_text_for_functions(cert.state.report_txt_path, filtered_functions)
        # Check target text
        st_matches = scan_text_for_functions(cert.state.st_txt_path, filtered_functions)
        
        # Combine matches
        all_matches = {**report_matches}
        for k, v in st_matches.items():
            all_matches[k] = all_matches.get(k, 0) + v
            
        if all_matches:
            results.append({
                'Dataset': 'CC',
                'ID': cert.dgst,
                'Manufacturer': cert.manufacturer,
                'Name': cert.name,
                'IssueDate': cert.not_valid_before,
                'MatchedFunctions': list(all_matches.keys()),
                'FunctionCount': len(all_matches),
                'TotalOccurrences': sum(all_matches.values())
            })
print(f"Found {len(results)} CC certificates mentioning OpenSSL functions.")

### 5. Scan Full FIPS 140 Dataset

In [ ]:
print("Scanning FIPS Dataset...")
for cert in fips_dataset:
    if not mentions_openssl_fips(cert):
        policy_matches = scan_text_for_functions(cert.state.sp.txt_path, filtered_functions)
        if policy_matches:
            results.append({
                "Dataset": "FIPS",
                "ID": cert.dgst,
                "Manufacturer": cert.web_data.vendor if hasattr(cert, "web_data") and cert.web_data else None,
                "Name": cert.web_data.module_name if hasattr(cert, "web_data") and cert.web_data else None,
                "IssueDate": cert.web_data.validation_history[0].date if hasattr(cert, "web_data") and cert.web_data and cert.web_data.validation_history else None,
                "MatchedFunctions": list(policy_matches.keys()),
                "FunctionCount": len(policy_matches),
                "TotalOccurrences": sum(policy_matches.values())
            })
print(f"Found {len(results)} total certificates mentioning OpenSSL functions.")


### 6. Display Results and Export Matched Entries

In [ ]:
import pandas as pd
from pathlib import Path

df_undisclosed = pd.DataFrame(results)

if not df_undisclosed.empty:
    df_undisclosed = df_undisclosed.sort_values(by="FunctionCount", ascending=False)
    display(df_undisclosed.head(20))
    print(f"Found {len(df_undisclosed)} certificates using OpenSSL functions without mentioning OpenSSL.")
    
    output_dir = Path("./results")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    csv_path = output_dir / "undisclosed_rebranding_matched_entries.csv"
    json_path = output_dir / "undisclosed_rebranding_matched_entries.json"
    
    df_undisclosed.to_csv(csv_path, index=False)
    df_undisclosed.to_json(json_path, orient="records", indent=4)
    print(f"Exported matched entries to {csv_path} and {json_path}")
else:
    print("No undisclosed rebranding found.")
